# UMUD Challenge — OpenCV + MLP Pipeline

**Approach:** No deep CNN. Instead:
1. For every pixel, extract a small patch around it and flatten to a vector
2. A plain fully-connected MLP classifies each pixel as aponeurosis or not
3. OpenCV geometry (line fitting, structure tensor) extracts MT, PA, FL

**Requirements:** GPU T4 x2, Internet OFF is fine (no pretrained weights downloaded)

In [ ]:
# =====================================================================
# CELL 1 : imports + find the data folders
# =====================================================================
import os, random, math, glob
import numpy as np
import cv2
import torch, torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# --- Robust directory finder for Kaggle's nested structure ---
SEARCH_ROOTS = ['/kaggle/input', '/content', '.']

def find_dir(name):
    """Walk search roots to find a directory with the given name."""
    for r in SEARCH_ROOTS:
        if not os.path.exists(r):
            continue
        for cur, dirs, _ in os.walk(r):
            if name in dirs:
                return os.path.join(cur, name)
    return None

APO_IMG_DIR  = find_dir('apo_images_new_model_v1')
APO_MSK_DIR  = find_dir('apo_masks_new_model_v1')
FASC_IMG_DIR = find_dir('fasc_images_new_model_v1')
FASC_MSK_DIR = find_dir('fasc_masks_new_model_v1')
TEST_IMG_DIR = find_dir('test_set_v2')

print('APO images :', APO_IMG_DIR)
print('APO masks  :', APO_MSK_DIR)
print('FASC images:', FASC_IMG_DIR)
print('FASC masks :', FASC_MSK_DIR)
print('TEST images:', TEST_IMG_DIR)

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)

def list_images(folder):
    """List only valid image files, filtering out .DS_Store, thumbnails, etc."""
    ok = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
    return sorted(f for f in os.listdir(folder)
                  if f.lower().endswith(ok) and not f.startswith('.'))

# --- Find sample submission (for matching image_id format) ---
SAMPLE_CSV = None
for r in SEARCH_ROOTS:
    if not os.path.exists(r):
        continue
    for cur, _, files in os.walk(r):
        for f in files:
            if 'sample' in f.lower() and 'submission' in f.lower() and f.endswith('.csv'):
                SAMPLE_CSV = os.path.join(cur, f)
                break
        if SAMPLE_CSV:
            break
    if SAMPLE_CSV:
        break
print('Sample CSV :', SAMPLE_CSV)

if SAMPLE_CSV:
    # Detect delimiter (this competition uses semicolons)
    with open(SAMPLE_CSV, 'r') as fh:
        header = fh.readline()
    SEP = ';' if ';' in header else ','
    sample_df = pd.read_csv(SAMPLE_CSV, sep=SEP)
    print(f'Sample submission: {len(sample_df)} rows, sep="{SEP}"')
    print(sample_df.head())

In [ ]:
# =====================================================================
# CELL 2 : PRE-PROCESSING
# =====================================================================
# Working size — MLP can't afford full resolution, keep 3:2 ratio
W, H = 288, 192

# Optional crop (fractions of ORIGINAL image) to cut text/scale bars
CROP = dict(top=0.00, bottom=1.00, left=0.00, right=1.00)

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def read_gray(path):
    """Read any image (8 or 16 bit tif/png) as uint8 grayscale."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f'Cannot read: {path}')
    if img.ndim == 3:
        img = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2GRAY)
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return img

def crop_frac(img):
    h, w = img.shape[:2]
    y0, y1 = int(CROP['top'] * h), int(CROP['bottom'] * h)
    x0, x1 = int(CROP['left'] * w), int(CROP['right'] * w)
    return img[y0:y1, x0:x1]

def preprocess_image(path):
    """
    raw file -> float32 image (H,W), standardised (mean 0, std 1).
    Steps: gray -> crop -> resize -> CLAHE (local contrast) -> z-score.
    Also returns the 8-bit version for display.
    """
    g = crop_frac(read_gray(path))
    g = cv2.resize(g, (W, H), interpolation=cv2.INTER_AREA)
    g = _clahe.apply(g)
    x = g.astype(np.float32) / 255.0
    x = (x - x.mean()) / (x.std() + 1e-6)
    return x, g

def preprocess_mask(path):
    """
    raw mask -> binary float32 (H,W) with 1 = aponeurosis.
    Fixes inverted masks (white background) automatically.
    """
    m = read_gray(path)
    m = (m > 127).astype(np.uint8)
    if m.mean() > 0.5:                      # background was white -> invert
        m = 1 - m
    m = crop_frac(m * 255)
    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.float32)

# ---- Quick visual check ----
if APO_IMG_DIR and APO_MSK_DIR:
    common = sorted(set(list_images(APO_IMG_DIR)) & set(list_images(APO_MSK_DIR)))
    print(f'Image/mask pairs found: {len(common)}')
    f = common[0]
    raw  = read_gray(os.path.join(APO_IMG_DIR, f))
    x, g = preprocess_image(os.path.join(APO_IMG_DIR, f))
    m    = preprocess_mask(os.path.join(APO_MSK_DIR, f))
    print(f'raw size {raw.shape[::-1]}  ->  working size {(W, H)}')
    print(f'mask foreground pixels: {int(m.sum())} / {m.size} ({100*m.mean():.2f}%)')

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].imshow(raw, cmap='gray'); ax[0].set_title('raw')
    ax[1].imshow(g,   cmap='gray'); ax[1].set_title(f'pre-processed {W}x{H} (CLAHE)')
    ax[2].imshow(m,   cmap='gray'); ax[2].set_title('mask (1 = aponeurosis)')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
else:
    common = []
    print('WARNING: APO image or mask dir not found!')

In [ ]:
# =====================================================================
# CELL 3 : turn pixels into training samples (patches)
# =====================================================================
P = 15                      # patch size (odd). Each pixel -> P*P numbers
HALF = P // 2

def get_patches(x, ys, xs):
    """x: (H,W) image. Returns (N, P*P + 2) features for pixels (ys,xs).
    Last 2 features = normalised row / column position (aponeuroses are
    always at similar depths, so position helps a plain MLP a lot)."""
    padded = np.pad(x, HALF, mode='reflect')
    win = sliding_window_view(padded, (P, P))          # (H, W, P, P) view
    patches = win[ys, xs].reshape(len(ys), -1)
    pos = np.stack([ys / (H - 1), xs / (W - 1)], axis=1).astype(np.float32)
    return np.concatenate([patches, pos], axis=1).astype(np.float32)

def sample_from_pair(x, m, n=3000):
    """Half the samples from aponeurosis pixels, half from background."""
    pos = np.argwhere(m > 0.5)
    neg = np.argwhere(m < 0.5)
    if len(pos) == 0:
        return None
    k = n // 2
    p = pos[np.random.choice(len(pos), k, replace=len(pos) < k)]
    q = neg[np.random.choice(len(neg), k, replace=len(neg) < k)]
    idx = np.concatenate([p, q])
    ys, xs = idx[:, 0], idx[:, 1]
    return get_patches(x, ys, xs), m[ys, xs]

# Use all available pairs for training, keep 20 for validation
N_VAL_IMG = 20
random.shuffle(common)
train_files = common[:-N_VAL_IMG] if len(common) > N_VAL_IMG else common
val_files   = common[-N_VAL_IMG:] if len(common) > N_VAL_IMG else common[:5]
print(f'Train images: {len(train_files)}, Val images: {len(val_files)}')

Xs, Ys = [], []
for i, f in enumerate(train_files):
    x = preprocess_image(os.path.join(APO_IMG_DIR, f))[0]
    m = preprocess_mask(os.path.join(APO_MSK_DIR, f))
    out = sample_from_pair(x, m)
    if out is not None:
        Xs.append(out[0]); Ys.append(out[1])
    if (i + 1) % 100 == 0:
        print(f'  prepared {i+1}/{len(train_files)} images')

X_train = torch.tensor(np.concatenate(Xs))
y_train = torch.tensor(np.concatenate(Ys)).unsqueeze(1)
print(f'Training samples: {tuple(X_train.shape)}  positives: {float(y_train.mean()):.3f}')
print(f'Feature dim: {X_train.shape[1]} ({P}x{P} patch + 2 position features)')

In [ ]:
# =====================================================================
# CELL 4 : the basic neural network (fully connected, no CNN)
# =====================================================================
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),   nn.ReLU(),
            nn.Linear(64, 1))                 # 1 logit per pixel
    def forward(self, x):
        return self.net(x)

model = MLP(X_train.shape[1]).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

@torch.no_grad()
def predict_map(model, x):
    """x: preprocessed (H,W) image -> probability map (H,W) in [0,1]."""
    model.eval()
    ys, xs = np.divmod(np.arange(H * W), W)
    feats = get_patches(x, ys, xs)
    probs = []
    for i in range(0, len(feats), 8192):
        b = torch.tensor(feats[i:i + 8192]).to(DEVICE)
        probs.append(torch.sigmoid(model(b)).cpu().numpy())
    return np.concatenate(probs).reshape(H, W)

def dice(pred_bin, gt):
    inter = (pred_bin * gt).sum()
    return (2 * inter + 1e-6) / (pred_bin.sum() + gt.sum() + 1e-6)

def validate(thresh=0.5):
    ds = []
    for f in val_files:
        x = preprocess_image(os.path.join(APO_IMG_DIR, f))[0]
        m = preprocess_mask(os.path.join(APO_MSK_DIR, f))
        ds.append(dice((predict_map(model, x) > thresh).astype(np.float32), m))
    return float(np.mean(ds))

In [ ]:
# =====================================================================
# CELL 5 : train the MLP
# =====================================================================
EPOCHS, BS, LR = 15, 1024, 1e-3
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.BCEWithLogitsLoss()

best_dice = 0.0
ckpt_path = os.path.join(OUT_DIR, 'mlp_apo.pth')

for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(X_train))
    total = 0.0
    for i in range(0, len(perm), BS):
        idx = perm[i:i + BS]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(idx)
    scheduler.step()
    
    val_dice = validate()
    print(f'epoch {ep:02d} | loss {total/len(perm):.4f} | val Dice {val_dice:.3f}', end='')
    
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), ckpt_path)
        print(f'  <-- best (saved)', end='')
    print()

# Reload best
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
print(f'\nBest validation Dice: {best_dice:.3f}')

In [ ]:
# =====================================================================
# CELL 6 : show predicted aponeurosis maps
# =====================================================================
def show_prediction(path, thresh=0.5, save_name=None):
    x, g = preprocess_image(path)
    prob = predict_map(model, x)                       # (H,W) 0..1
    binary = (prob > thresh).astype(np.uint8) * 255

    # OpenCV colouring
    heat    = cv2.applyColorMap((prob * 255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    overlay[binary > 0] = (0, 0, 255)                  # red = predicted apo (BGR)
    overlay = cv2.addWeighted(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR), 0.6, overlay, 0.4, 0)

    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    ax[0].imshow(g, cmap='gray');                       ax[0].set_title('input (pre-processed)')
    ax[1].imshow(cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)); ax[1].set_title('probability map')
    ax[2].imshow(binary, cmap='gray');                  ax[2].set_title(f'binary (> {thresh})')
    ax[3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax[3].set_title('overlay on image')
    for a in ax: a.axis('off')
    plt.suptitle(os.path.basename(path)); plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUT_DIR, save_name), dpi=100)
    plt.show()

if TEST_IMG_DIR:
    test_files = list_images(TEST_IMG_DIR)
    print(f'Test images found: {len(test_files)}')
    for f in test_files[:3]:
        show_prediction(os.path.join(TEST_IMG_DIR, f), save_name=f'pred_{f}.png')
else:
    print('No test folder found - showing a validation image instead.')
    show_prediction(os.path.join(APO_IMG_DIR, val_files[0]))

In [ ]:
# =====================================================================
# CELL 7 : mm-per-pixel scale
# =====================================================================
# No scale CSV exists in this dataset, so we use resolution-based
# estimates. These are rough but consistent across similar scanners.

# Search for any scale CSV (in case user attaches one as extra data)
SCALE_CSV = None
for r in SEARCH_ROOTS:
    if not os.path.exists(r):
        continue
    for cur, _, files in os.walk(r):
        for f in files:
            if 'scale' in f.lower() and f.lower().endswith('.csv'):
                SCALE_CSV = os.path.join(cur, f)
                break
        if SCALE_CSV: break
    if SCALE_CSV: break
print('Scale CSV:', SCALE_CSV)

SCALE_MAP = {}
if SCALE_CSV:
    sdf = pd.read_csv(SCALE_CSV)
    print('columns:', list(sdf.columns))
    print(sdf.head(3).to_string())
    for _, row in sdf.iterrows():
        key = str(row.get('record_id', row.iloc[0])).strip()
        key = os.path.splitext(key)[0].lower()
        v = None
        for col in ('apo_mm_per_pixel', 'fasc_mm_per_pixel'):
            if col in sdf.columns and pd.notna(row[col]) and float(row[col]) > 0:
                v = float(row[col]); break
        if v is None:
            for col in ('apo_spacing', 'fasc_spacing'):
                if col in sdf.columns and pd.notna(row[col]) and float(row[col]) > 0:
                    v = 10.0 / float(row[col]); break
        if v is None:
            for col in ('apo_depth_mm', 'fasc_depth_mm'):
                if col in sdf.columns and pd.notna(row[col]) and float(row[col]) > 0:
                    v = ('depth', float(row[col])); break
        if v is not None:
            SCALE_MAP[key] = v
    print(f'Loaded {len(SCALE_MAP)} scale entries from CSV')

def fallback_mmpx(w, h):
    """Resolution-based mm/pixel estimate when no CSV is available."""
    # Common US scanner settings by resolution cluster
    if (w, h) == (1200, 800):  return 50.0 / 720.0     # ~0.0694
    if (w, h) == (1640, 1080): return 45.0 / 980.0     # ~0.0459
    if (w, h) == (760, 500):   return 50.0 / 450.0     # ~0.1111
    if (w, h) == (512, 512):   return 40.0 / 460.0     # ~0.0870
    if (w, h) == (800, 652):   return 50.0 / 590.0     # ~0.0847
    if (w, h) == (800, 600):   return 50.0 / 540.0     # ~0.0926
    if (w, h) == (660, 556):   return 45.0 / 500.0     # ~0.0900
    if (w, h) == (1234, 688):  return 45.0 / 620.0     # ~0.0726
    # Generic fallback: assume ~50mm depth mapped to ~90% of image height
    return 50.0 / max(h * 0.90, 1)

def get_mmpx(fname, w, h):
    """Returns (mm_per_pixel, source) where source is 'csv','depth','fallback'."""
    v = SCALE_MAP.get(os.path.splitext(fname)[0].lower())
    if v is None:
        return fallback_mmpx(w, h), 'fallback'
    if isinstance(v, tuple):
        return v[1] / max(h * 0.90, 1), 'depth'
    return v, 'csv'

# Check how many test images get a real vs fallback scale
if TEST_IMG_DIR:
    srcs = []
    for f in list_images(TEST_IMG_DIR):
        raw = read_gray(os.path.join(TEST_IMG_DIR, f))
        h_raw, w_raw = raw.shape
        srcs.append(get_mmpx(f, w_raw, h_raw)[1])
    print('\nScale source distribution:')
    print(pd.Series(srcs).value_counts().to_string())

In [ ]:
# =====================================================================
# CELL 8 : geometry -> MT (mm), PA (deg), FL (mm)
# =====================================================================
THRESH = 0.5            # probability threshold for the aponeurosis map

def crop_scale(orig_w, orig_h):
    """working-pixel -> original-pixel factors (accounts for crop+resize)."""
    cw = (CROP['right'] - CROP['left']) * orig_w
    ch = (CROP['bottom'] - CROP['top']) * orig_h
    return cw / W, ch / H          # sx, sy

def fit_two_lines(prob):
    """Two biggest wide blobs -> (sup_line, dep_line, x_lo, x_hi).
    Lines are (m,b) with y = m*x + b in WORKING pixels. None if not found."""
    b = (prob > THRESH).astype(np.uint8)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lbl, stats, cent = cv2.connectedComponentsWithStats(b, connectivity=8)
    
    # Need blobs that span at least 25% of image width
    cands = [i for i in range(1, n) if stats[i, cv2.CC_STAT_WIDTH] >= 0.25 * W]
    if len(cands) < 2:
        return None
    # Keep the 2 largest candidates, sorted top-to-bottom
    cands = sorted(cands, key=lambda i: stats[i, cv2.CC_STAT_AREA], reverse=True)[:2]
    cands = sorted(cands, key=lambda i: cent[i][1])      # top = superficial
    
    lines, xr = [], []
    for i in cands:
        ys, xs = np.nonzero(lbl == i)
        pts = np.column_stack([xs, ys]).astype(np.float32).reshape(-1, 1, 2)
        vx, vy, x0, y0 = cv2.fitLine(pts, cv2.DIST_HUBER, 0, 0.01, 0.01).flatten()
        if abs(vx) < 1e-6:
            return None
        m = float(vy / vx)
        lines.append((m, float(y0 - m * x0)))
        xr.append((xs.min(), xs.max()))
    
    x_lo = max(xr[0][0], xr[1][0])
    x_hi = min(xr[0][1], xr[1][1])
    if x_hi - x_lo < 0.2 * W:
        return None
    return lines[0], lines[1], x_lo, x_hi

def thickness_px(sup, dep, x_lo, x_hi, sx, sy):
    """Mean perpendicular distance between two lines, in ORIGINAL pixels."""
    xs = np.linspace(x_lo, x_hi, 200)
    vert = (dep[0] * xs + dep[1]) - (sup[0] * xs + sup[1])   # working px
    if vert.mean() <= 0:
        return None
    m_dep_o = dep[0] * sy / sx                               # slope in original px
    return float(vert.mean() * sy / math.sqrt(1 + m_dep_o ** 2))

def fascicle_angle(x, sup, dep, mt_w, sx, sy):
    """Structure tensor inside the band between the aponeuroses.
    Returns angle (deg) between fascicle direction and deep aponeurosis."""
    bl = cv2.GaussianBlur(x, (0, 0), 1.0)
    gx = cv2.Sobel(bl, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3)
    Jxx = cv2.GaussianBlur(gx * gx, (0, 0), 4)
    Jyy = cv2.GaussianBlur(gy * gy, (0, 0), 4)
    Jxy = cv2.GaussianBlur(gx * gy, (0, 0), 4)
    coh = np.sqrt((Jxx - Jyy) ** 2 + 4 * Jxy ** 2) / (Jxx + Jyy + 1e-8)
    XS, YS = np.meshgrid(np.arange(W, dtype=np.float32),
                         np.arange(H, dtype=np.float32))
    ys_line = sup[0] * XS + sup[1]
    yd_line = dep[0] * XS + dep[1]
    m_dep_o = dep[0] * sy / sx
    theta_deep = math.degrees(math.atan(m_dep_o))
    
    for mf in (0.10, 0.05, 0.15):                    # band margins as fraction of MT
        mg = mt_w * mf
        band = (YS >= np.minimum(ys_line, yd_line) + mg) & \
               (YS <= np.maximum(ys_line, yd_line) - mg)
        if band.sum() < 300:
            continue
        w = np.where(band, coh, 0.0); tw = w.sum()
        if tw < 1e-6:
            continue
        a = (w * Jxx).sum() / tw
        c = (w * Jyy).sum() / tw
        bxy = (w * Jxy).sum() / tw
        th_g = 0.5 * math.atan2(2 * bxy, a - c)      # gradient direction
        th_f = th_g + math.pi / 2                    # fascicle direction (working px)
        # working px -> original px (undo anisotropic resize)
        th_f_o = math.degrees(math.atan2(math.sin(th_f) * sy,
                                         math.cos(th_f) * sx))
        d = (th_f_o - theta_deep + 90) % 180 - 90    # wrap to [-90, 90)
        pa = abs(d)
        if 3.0 <= pa <= 60.0:
            return pa
    return None

def measure(prob, x, orig_w, orig_h, mmpx):
    """-> dict(mt_mm, pa, fl_mm, lines). Values can be None if failed."""
    out = dict(mt_mm=None, pa=None, fl_mm=None, lines=None)
    res = fit_two_lines(prob)
    if res is None:
        return out
    sup, dep, x_lo, x_hi = res
    sx, sy = crop_scale(orig_w, orig_h)
    mt_px = thickness_px(sup, dep, x_lo, x_hi, sx, sy)
    if mt_px is None:
        return out
    out['mt_mm'] = mt_px * mmpx
    out['lines'] = (sup, dep, x_lo, x_hi)
    mt_w = mt_px / sy   # MT in working pixels
    out['pa'] = fascicle_angle(x, sup, dep, mt_w, sx, sy)
    # FL = MT / sin(PA) if both available
    if out['pa'] is not None and out['pa'] > 2.0:
        out['fl_mm'] = out['mt_mm'] / math.sin(math.radians(out['pa']))
    return out

print('Geometry functions defined.')

In [ ]:
# =====================================================================
# CELL 9 : run on every test image, build submission.csv
# =====================================================================
test_files = list_images(TEST_IMG_DIR)
print(f'Processing {len(test_files)} test images...')

recs = {}
for i, f in enumerate(test_files):
    path = os.path.join(TEST_IMG_DIR, f)
    raw = read_gray(path)
    h_raw, w_raw = raw.shape                          # ORIGINAL size
    mmpx, src = get_mmpx(f, w_raw, h_raw)
    x, g = preprocess_image(path)
    prob = predict_map(model, x)
    r = measure(prob, x, w_raw, h_raw, mmpx)
    r.update(mmpx=mmpx, src=src, g=g, prob=prob)
    recs[f] = r
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(test_files)}')

# Collect successful measurements for computing medians
mt_vals = np.array([r['mt_mm'] if r['mt_mm'] is not None else np.nan for r in recs.values()])
pa_vals = np.array([r['pa'] if r['pa'] is not None else np.nan for r in recs.values()])
fl_vals = np.array([r['fl_mm'] if r['fl_mm'] is not None else np.nan for r in recs.values()])

print(f'\nMT found: {np.isfinite(mt_vals).sum()}/{len(mt_vals)}')
print(f'PA found: {np.isfinite(pa_vals).sum()}/{len(pa_vals)}')
print(f'FL found: {np.isfinite(fl_vals).sum()}/{len(fl_vals)}')

# Failed images get the median of successful ones (not a fake constant)
MT_FILL = float(np.nanmedian(mt_vals)) if np.isfinite(mt_vals).any() else 19.0
PA_FILL = float(np.nanmedian(pa_vals)) if np.isfinite(pa_vals).any() else 15.0
FL_FILL = MT_FILL / math.sin(math.radians(PA_FILL))  # geometrically consistent
print(f'\nFallback values -> MT {MT_FILL:.2f} mm, PA {PA_FILL:.2f} deg, FL {FL_FILL:.2f} mm')
print(f'FL consistency check: MT/sin(PA) = {MT_FILL / math.sin(math.radians(PA_FILL)):.2f} mm')

# Build the results table
rows = []
for f, r in recs.items():
    mt_mm = r['mt_mm'] if r['mt_mm'] is not None else MT_FILL
    pa_d  = r['pa']    if r['pa']    is not None else PA_FILL
    fl_mm = r['fl_mm'] if r['fl_mm'] is not None else None
    
    # Physiologically sane limits
    mt_mm = float(np.clip(mt_mm, 5, 60))
    pa_d  = float(np.clip(pa_d, 3, 45))
    
    # FL = MT / sin(PA) — geometrically consistent
    fl_computed = mt_mm / math.sin(math.radians(pa_d))
    if fl_mm is not None:
        # Blend measured and geometric FL
        fl_mm = 0.5 * float(np.clip(fl_mm, 20, 200)) + 0.5 * fl_computed
    else:
        fl_mm = fl_computed
    fl_mm = float(np.clip(fl_mm, 20, 200))
    
    rows.append(dict(
        key=f,
        pa_deg=round(pa_d, 3),
        fl_mm=round(fl_mm, 3),
        mt_mm=round(mt_mm, 3)
    ))

res_df = pd.DataFrame(rows)

# Match image_id format from sample submission
if SAMPLE_CSV and len(sample_df) >= len(test_files):
    ids = sample_df['image_id'].astype(str).tolist()
    by_name = {r['key']: r for r in rows}
    by_stem = {os.path.splitext(r['key'])[0]: r for r in rows}
    out_rows = []
    missing = []
    for img_id in ids:
        r = by_name.get(img_id) or by_stem.get(os.path.splitext(img_id)[0])
        if r is None:
            missing.append(img_id)
            # Use global fallback for missing images
            out_rows.append(dict(
                image_id=img_id,
                pa_deg=round(PA_FILL, 3),
                fl_mm=round(FL_FILL, 3),
                mt_mm=round(MT_FILL, 3)
            ))
        else:
            out_rows.append(dict(
                image_id=img_id,
                pa_deg=r['pa_deg'],
                fl_mm=r['fl_mm'],
                mt_mm=r['mt_mm']
            ))
    if missing:
        print(f'WARNING: {len(missing)} image_ids from sample submission had no prediction: {missing[:5]}')
    sub = pd.DataFrame(out_rows)
else:
    sub = res_df.rename(columns={'key': 'image_id'})[['image_id', 'pa_deg', 'fl_mm', 'mt_mm']].sort_values('image_id').reset_index(drop=True)

assert sub['image_id'].is_unique, 'Duplicate image_ids!'
assert not sub.isna().any().any(), 'NaN values in submission!'

# Save with semicolons to match sample submission format
SUB_PATH = os.path.join(OUT_DIR, 'submission.csv')
sub.to_csv(SUB_PATH, index=False, sep=';')
print(f'\nSaved {SUB_PATH}  ({len(sub)} rows, sep=";")')
print(sub.describe().to_string())
print()
print(sub.head(10))
print(f'\nSanity: quadriceps MT ~15-35 mm, PA ~8-25 deg, FL ~50-130 mm.')

In [ ]:
# =====================================================================
# CELL 10 : visual check — are the 2 fitted lines on the aponeuroses?
# =====================================================================
def draw_check(f):
    r = recs[f]
    g = cv2.cvtColor(r['g'], cv2.COLOR_GRAY2BGR)
    title_parts = [f]
    
    if r['lines'] is not None:
        sup, dep, x_lo, x_hi = r['lines']
        for (m, b), col in ((sup, (0, 255, 255)), (dep, (255, 0, 255))):
            pt1 = (int(x_lo), int(m * x_lo + b))
            pt2 = (int(x_hi), int(m * x_hi + b))
            cv2.line(g, pt1, pt2, col, 1)
    
    # Find this file's row in submission
    stem = os.path.splitext(f)[0]
    match = sub[sub['image_id'].str.contains(stem, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        title_parts.append(f'MT={row.mt_mm:.1f}mm  PA={row.pa_deg:.1f}°  FL={row.fl_mm:.1f}mm')
    title_parts.append(f'[scale {r["mmpx"]:.4f} mm/px from {r["src"]}]')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(g, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Fitted aponeurosis lines')
    axes[0].axis('off')
    
    # Show probability map too
    heat = cv2.applyColorMap((r['prob'] * 255).astype(np.uint8), cv2.COLORMAP_JET)
    axes[1].imshow(cv2.cvtColor(heat, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Apo probability map')
    axes[1].axis('off')
    
    plt.suptitle('  |  '.join(title_parts), fontsize=10)
    plt.tight_layout(); plt.show()

# Show first 6 test images
for f in test_files[:6]:
    draw_check(f)

print('\n--- Pipeline complete! ---')
print(f'Submission file: {SUB_PATH}')